## Process
- Load CNAF file
- Clean data and first mapping for bdd injection
- Drop duplicates
- Add default column values
- Output to CSV

## Encoding
CNAF -> ascii

In [ ]:
import os
import pandas as pd
import json
import numpy as np
import csv
import re

from datetime import datetime
import sys
from pathlib import Path

try:
    from utils.data_utils import unaccent_and_upper, format_insee_or_postal_code
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break
    from utils.data_utils import unaccent_and_upper, format_insee_or_postal_code
from dotenv import load_dotenv

load_dotenv()

cnaf_input_filepath = os.environ['CNAF_PATHFILE_2026']
base_output_filepath = os.environ['DB_CNAF_EXPORT_2026']
qf_batch_input_filepath = os.environ['QF_BATCH_INPUT_PATHFILE_2026']
qf_batch_output_filepath = os.environ['QF_BATCH_OUTPUT_PATHFILE_2026']

# INSEE COG countries & territories, used to turn CNAF's PAYSNAIDOS label into a COG code.
# Defaults to the copy shipped next to this notebook when the env var isn't set.
cog_pays_input_filepath = os.environ.get(
    'COG_PAYS_PATHFILE_2026', str(Path.cwd() / 'v_pays_territoire_2026.csv'))


In [ ]:
# CNAF
cnaf_column_type = {
    'CODORG': 'str',
    'MATRICULE': 'str',
    'QUALDOS': 'str',
    'RESPDOS': 'str',
    'NOMNAIDOS': 'str',
    'PRENOMDOS': 'str',
    'DTNAIDOS': 'str',
    'SEXDOS': 'str',
    'COMMUNENAIDOS': 'str',
    'PAYSNAIDOS': 'str',
    'ORIGINESELECTION': 'str',
    'NOMCOMPLET': 'str',
    'ADRLIG1DESTDOS': 'str',
    'ADRLIG2DESTDOS': 'str',
    'ADRLIG3DESTDOS': 'str',
    'ADRLIG4DESTDOS': 'str',
    'ADRLIG5DESTDOS': 'str',
    'ADRLIG6DESTDOS': 'str',
    'NUMINSEE': 'str',
    'ADRMAIL': 'str',
    'NUMTEL': 'str',
    'NOMENF': 'str',
    'PRENOMENF': 'str',
    'DTNAIENF': 'str',
    'SEXENF': 'str',
}

cnaf_df = pd.read_csv(cnaf_input_filepath, encoding='ascii', on_bad_lines='skip', sep=';', quoting=csv.QUOTE_NONE, dtype=cnaf_column_type, engine="c", keep_default_na=False, header=1)


In [ ]:
# delete last row (it is not a valid row)
cnaf_df = cnaf_df.iloc[:-1]

In [ ]:
# Clean white spaces within all columns
for col in list(cnaf_df.columns):
  cnaf_df[col] = cnaf_df[col].str.strip()

In [ ]:
# Explode postal code & commune from initial column containing both
cnaf_df[['CODE_POSTAL', 'COMMUNE']] = cnaf_df['ADRLIG5DESTDOS'].str.split(' ', n=1, expand=True)
cnaf_df[['CODE_POSTAL', 'COMMUNE']] = cnaf_df[['CODE_POSTAL', 'COMMUNE']].transform(lambda x: x.str.strip())

In [ ]:
# Clean extra white spaces
cnaf_df['NOMCOMPLET'] = cnaf_df['NOMCOMPLET'].astype(str).str.replace(r'\s+', ' ', regex=True)

In [ ]:
# map CNAF
cnaf_column_mapping = {
    # infos about allocataire
    'MATRICULE': 'allocataire-matricule',
    'CODORG': 'allocataire-code_organisme',
    'QUALDOS': 'allocataire-qualite',
    'RESPDOS': 'allocataire-nom',
    'PRENOMDOS': 'allocataire-prenom',
    'ADRMAIL': 'allocataire-courriel',
    'NUMTEL': 'allocataire-telephone',

    # allocataire pivot identity for the quotient_familial API call - filled by CNAF only
    # on ARS-origin rows, kept out of the site-facing 'allocataire-nom'/'allocataire-prenom'
    'NOMNAIDOS': 'allocataire-nom_naissance',
    'DTNAIDOS': 'allocataire-date_naissance',
    'SEXDOS': 'allocataire-genre',
    'COMMUNENAIDOS': 'allocataire-code_insee_naissance',
    'PAYSNAIDOS': 'allocataire-pays_naissance',

    # CNAF's own AAH/ARS/AEEH classification, replaces the previous DOB+name heuristic
    'ORIGINESELECTION': 'situation_origine',

    # adresse allocataire
    'CODE_POSTAL': 'adresse_allocataire-code_postal',
    'COMMUNE': 'adresse_allocataire-commune',
    'NUMINSEE': 'adresse_allocataire-code_insee',

    # infos about beneficiary
    'DTNAIENF': 'date_naissance',
    'SEXENF': 'genre',
    'NOMENF': 'nom',
    'PRENOMENF': 'prenom',
}

df_psp_mapped_cnaf = cnaf_df.copy()
df_psp_mapped_cnaf = df_psp_mapped_cnaf.rename(columns=cnaf_column_mapping)


In [ ]:
# Allocataire missing phone number
phone_replacements = {
    '0000000000': np.NaN,
    '0600000000': np.NaN,
    '0700000001': np.NaN,
    '0100000000': np.NaN,
    '0400000000': np.NaN,
    '0600000001': np.NaN,
    '0700000000': np.NaN
}

df_psp_mapped_cnaf['allocataire-telephone'] = df_psp_mapped_cnaf['allocataire-telephone'].replace(phone_replacements)

# Allocataire's qualite
df_psp_mapped_cnaf['allocataire-qualite'] = df_psp_mapped_cnaf['allocataire-qualite'].str.strip().replace(
    {'MME': 'Mme', 'MR': 'M'})

# Additionnal address details
df_psp_mapped_cnaf['adresse_allocataire-cplt_adresse'] = (
    df_psp_mapped_cnaf['ADRLIG1DESTDOS'] + ' ' + df_psp_mapped_cnaf['ADRLIG2DESTDOS']).str.strip()

# Allocataire's street address
df_psp_mapped_cnaf['adresse_allocataire-voie'] = (
    df_psp_mapped_cnaf['ADRLIG3DESTDOS'] + ' ' + df_psp_mapped_cnaf['ADRLIG4DESTDOS']).str.strip()

# Organism & situation - CNAF now flags the category itself, no more DOB+name guessing
df_psp_mapped_cnaf['organisme'] = 'CAF'
situation_by_origin = {'ARS': 'jeune', 'AAH': 'AAH', 'AEEH': 'AEEH'}
df_psp_mapped_cnaf['situation'] = df_psp_mapped_cnaf['situation_origine'].map(situation_by_origin)


In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_cnaf['date_naissance'] = pd.to_datetime(df_psp_mapped_cnaf['date_naissance'], format='%d/%m/%Y')


In [ ]:
# Build the qf-batch input: only ARS-origin rows carry the allocataire pivot identity
# needed for quotient_familial. One call per household, not per child, since several
# beneficiary rows can share the same allocataire.

# QF route age window: 6-17 ans révolus. Applied here so we never spend a
# quotient_familial call on a household with no child in the window - the same
# bounds are re-applied per beneficiary row further down (an eligible household
# can also hold children outside the window).
QF_DOB_MIN = datetime(2009, 1, 1)
QF_DOB_MAX = datetime(2020, 12, 31)

# TODO matricule + code org en pivot suffisant ? A check en db
mask_qf_route = (
    (df_psp_mapped_cnaf['situation_origine'] == 'ARS')
    & (df_psp_mapped_cnaf['date_naissance'] >= QF_DOB_MIN)
    & (df_psp_mapped_cnaf['date_naissance'] <= QF_DOB_MAX)
)
df_qf_route = df_psp_mapped_cnaf[mask_qf_route]
df_qf_allocataires = df_qf_route.drop_duplicates(
    subset=['allocataire-code_organisme', 'allocataire-matricule']).copy()

df_qf_allocataires['allocataire-nom_usage'] = df_qf_allocataires['allocataire-nom']
df_qf_allocataires['allocataire-genre'] = df_qf_allocataires['allocataire-genre'].map({'M': 'male', 'F': 'female'})
df_qf_allocataires['allocataire-date_naissance'] = pd.to_datetime(
    df_qf_allocataires['allocataire-date_naissance'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')


# code_pays_naissance: PAYSNAIDOS holds the country *label* (FRANCE, MAROC, PORTUGAL...),
# not a code - mapped here to its INSEE COG code through v_pays_territoire (France -> 99100,
# Maroc -> 99350...). Labels are compared unaccented/uppercased, since the CNAF file is
# ascii-encoded while the COG file is not.
FRANCE_COG = '99100'


def normalize_country_label(label) -> str:
    # punctuation -> space *before* the ascii fold, which would otherwise drop the
    # apostrophe of "Côte d’Ivoire" and glue the two words together
    return re.sub(r'\s+', ' ', unaccent_and_upper(re.sub(r'[^\w]+', ' ', str(label)))).strip()


df_cog_pays = pd.read_csv(cog_pays_input_filepath, dtype=str, keep_default_na=False)
# LIBCOG first, LIBENR ("Royaume du Maroc") as a fallback spelling for the same COG.
cog_by_country_label = pd.concat([
    df_cog_pays[['LIBCOG', 'COG']].rename(columns={'LIBCOG': 'label'}),
    df_cog_pays[['LIBENR', 'COG']].rename(columns={'LIBENR': 'label'}),
])
cog_by_country_label['label'] = cog_by_country_label['label'].map(normalize_country_label)
cog_by_country_label = cog_by_country_label[cog_by_country_label['label'] != ''].drop_duplicates(
    subset='label', keep='first').set_index('label')['COG']

pays_naissance_label = df_qf_allocataires['allocataire-pays_naissance'].map(normalize_country_label)
df_qf_allocataires['allocataire-code_pays_naissance'] = pays_naissance_label.map(cog_by_country_label)

unmapped_labels = sorted(set(pays_naissance_label[
    (pays_naissance_label != '') & df_qf_allocataires['allocataire-code_pays_naissance'].isna()]))
if unmapped_labels:
    print(f"{len(unmapped_labels)} PAYSNAIDOS label(s) without a COG match: {unmapped_labels}")

# code_insee_naissance only makes sense for a birth in France: COMMUNENAIDOS holds a
# free-text foreign locality for the others, which is not an INSEE commune code. Dropped
# for every non-France COG (unmapped/empty labels included, since we can't assert France).
mask_born_abroad = df_qf_allocataires['allocataire-code_pays_naissance'] != FRANCE_COG
df_qf_allocataires.loc[mask_born_abroad, 'allocataire-code_insee_naissance'] = np.NaN
print(f"{int(mask_born_abroad.sum())} allocataire(s) born outside France: "
      "allocataire-code_insee_naissance cleared")


qf_batch_columns = [
    'allocataire-matricule',
    'allocataire-code_organisme',
    'allocataire-nom_naissance',
    'allocataire-nom_usage',
    'allocataire-prenom',
    'allocataire-date_naissance',
    'allocataire-genre',
    'allocataire-code_insee_naissance',
    'allocataire-code_pays_naissance',
]
df_qf_batch_input = df_qf_allocataires[qf_batch_columns]
df_qf_batch_input.to_csv(qf_batch_input_filepath, index=False, encoding='utf-8')

ars_rows = (df_psp_mapped_cnaf['situation_origine'] == 'ARS').sum()
unparsed_dob = df_qf_batch_input['allocataire-date_naissance'].isna().sum()
print(f"{len(df_qf_batch_input)} allocataire(s) (from {len(df_qf_route)} of {ars_rows} ARS row(s) "
      f"within the 6-17 ans window) written to {qf_batch_input_filepath} for qf-batch "
      f"({unparsed_dob} with an unparsed birthdate)")


## ⏸ Checkpoint: run qf-batch here
The cell above wrote `QF_BATCH_INPUT_PATHFILE_2026`. Run it through qf-batch.ts now
(detached, can take up to a week - see worker/src/scripts/qf-batch.ts), writing its
output to `QF_BATCH_OUTPUT_PATHFILE_2026`, before continuing past the next checkpoint
below. The cells in between only touch beneficiary-level data and don't need to wait.


In [ ]:
# remove unused 
df_psp_mapped_cnaf = df_psp_mapped_cnaf.drop(columns=[
    'NOMCOMPLET',
    'ADRLIG1DESTDOS',
    'ADRLIG2DESTDOS',
    'ADRLIG3DESTDOS',
    'ADRLIG4DESTDOS',
    'ADRLIG5DESTDOS',
    'ADRLIG6DESTDOS'
])

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot generate a code)
necessary_column = ['nom', 'prenom', 'date_naissance', 'genre']
df_valid_row = df_psp_mapped_cnaf.dropna(subset=necessary_column)

# remove columns with all null value
df_valid = df_valid_row.dropna(axis=1, how='all')

assert len(
    df_valid[df_psp_mapped_cnaf['nom'].isnull() | df_valid['prenom'].isnull() | df_valid['date_naissance'].isnull()]
) == 0

In [ ]:
# Upper case these columns for the merge
df_valid.loc[:,'prenom'] = df_valid['prenom'].astype(str).apply(unaccent_and_upper)
df_valid.loc[:,'nom'] = df_valid['nom'].astype(str).apply(unaccent_and_upper)
df_valid.loc[:,'genre'] = df_valid['genre'].astype(str).apply(unaccent_and_upper)

In [ ]:
# lower case on emails on all
df_valid.loc[:,'allocataire-courriel'] = df_valid['allocataire-courriel'].str.lower()

In [ ]:
# # Remove RGPD lines
# rgpd_emails = os.environ['RGPD_EMAILS_TO_EXCLUDE_PATHFILE']

# df_rgpd_emails = pd.read_csv(rgpd_emails)

# print(f"Number of beneficiaries before excluding RGPD lines : {len(df_valid)} ")

# df_valid = df_valid[~df_valid['allocataire-courriel'].str.lower().isin(df_rgpd_emails['email'].str.lower())]

# print(f"Number of beneficiaries after excluding RGPD lines : {len(df_valid)} ")

In [ ]:
# Preliminary filter, ahead of the precise QF/AAH/AEEH windows below: 1996-01-01 is the
# oldest birthdate any of the 3 routes can accept (AAH's lower bound).
mask_before = pd.to_datetime(df_valid['date_naissance']) >= datetime(1996, 1, 1)
df_valid_after = df_valid[mask_before]

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are outside all eligibility windows")


In [ ]:
from datetime import timedelta

# add missing 0 to phone numbers
mask_tel_not_null = df_valid_after['allocataire-telephone'].notna()
mask_no_zero_phone_number = ~df_valid_after.loc[mask_tel_not_null, 'allocataire-telephone'].str.startswith('0')
mask_9_char_phone = df_valid_after.loc[mask_tel_not_null, 'allocataire-telephone'].str.len() == 9
df_valid_after.loc[
    mask_tel_not_null & mask_no_zero_phone_number & mask_9_char_phone, 'allocataire-telephone'] = '0' + \
                                                                                                  df_valid_after[
                                                                                                      'allocataire-telephone']

# set '0' phone values to None
mask_tel_eq_zero = df_valid_after['allocataire-telephone'] == '0'
df_valid_after.loc[mask_tel_eq_zero, 'allocataire-telephone'] = np.NaN

In [ ]:
# set Nan values for not existing courriel
mask_email_not_existing = df_valid_after['allocataire-courriel'] == ''
df_valid_after.loc[mask_email_not_existing, 'allocataire-courriel'] = np.NaN

In [ ]:
# add 4h on all birthdates
df_valid_after['date_naissance'] = df_valid_after['date_naissance'] + timedelta(hours=4)

In [ ]:
# remove duplicate beneficiaries
df_valid_no_duplicate = df_valid_after.drop_duplicates(subset=[
    'date_naissance',
    'nom',
    'prenom',
    'genre',
    'organisme',
    'situation',
    'allocataire-qualite',
    'allocataire-matricule',
    'allocataire-code_organisme',
    'allocataire-telephone',
    'allocataire-nom',
    'allocataire-prenom',
    'allocataire-courriel',
])

print(f"{len(df_valid_after) - len(df_valid_no_duplicate)} duplicate rows were removed")

In [ ]:
# map allocataire json
def to_json_allocataire_without_null(row):
    allocataire_mapping = {
        'qualite': row['allocataire-qualite'],
        'matricule': row['allocataire-matricule'],
        'code_organisme': row['allocataire-code_organisme'],
        'telephone': row['allocataire-telephone'],
        'nom': unaccent_and_upper(row['allocataire-nom']),
        'prenom': unaccent_and_upper(row['allocataire-prenom']),
        'courriel': row['allocataire-courriel']
    }
    filtered_NaN_allocataire = {k: v for k, v in allocataire_mapping.items() if pd.notnull(v)}
    return json.dumps(filtered_NaN_allocataire, ensure_ascii=False)


df_valid_no_duplicate['allocataire'] = df_valid_no_duplicate.apply(to_json_allocataire_without_null, axis=1)

In [ ]:
# map adresse_allocataire json
def to_json_adresse_without_null(row):
    adresse_mapping = {
        'voie': row['adresse_allocataire-voie'],
        'code_postal': format_insee_or_postal_code(row['adresse_allocataire-code_postal']),
        'commune': row['adresse_allocataire-commune'],
        'code_insee': format_insee_or_postal_code(row['adresse_allocataire-code_insee']),
        'cplt_adresse': row['adresse_allocataire-cplt_adresse'],
    }

    filtered_address = {k: v for k, v in adresse_mapping.items() if pd.notnull(v)}
    return json.dumps(filtered_address, ensure_ascii=False)


df_valid_no_duplicate['adresse_allocataire'] = df_valid_no_duplicate.apply(to_json_adresse_without_null, axis=1)

In [ ]:
## drop null value
df_final = df_valid_no_duplicate.drop(columns=[
    'allocataire-qualite',
    'allocataire-matricule',
    'allocataire-code_organisme',
    'allocataire-nom',
    'allocataire-prenom',
    'allocataire-telephone',
    'allocataire-courriel',
    'adresse_allocataire-voie',
    'adresse_allocataire-code_postal',
    'adresse_allocataire-commune',
    'adresse_allocataire-code_insee',
    'adresse_allocataire-cplt_adresse',
    # qf-batch pivot-only columns, not part of the site-facing DB schema
    'situation_origine',
    'allocataire-nom_naissance',
    'allocataire-date_naissance',
    'allocataire-genre',
    'allocataire-code_insee_naissance',
    'allocataire-pays_naissance',
])


## ▶ Resume here once qf-batch has finished
The cell below reads `QF_BATCH_OUTPUT_PATHFILE_2026` and joins the verdict back onto
every beneficiary row of each allocataire.


In [ ]:
# qf-batch.ts runs out-of-band (can take up to a week) - read its verdict back in here.
# It records the raw quotient (qf_value), not an eligibility boolean: the threshold is
# applied below, in the QF route cell.
df_qf_batch_output = pd.read_csv(qf_batch_output_filepath, dtype=str, keep_default_na=False)
df_qf_batch_output['allocataire_key'] = (
    df_qf_batch_output['allocataire-code_organisme'] + '|' + df_qf_batch_output['allocataire-matricule'])
qf_value_by_allocataire = pd.to_numeric(
    df_qf_batch_output.set_index('allocataire_key')['qf_value'], errors='coerce')

# One quotient_familial value per household, fanned out to every child row of that
# allocataire. .map() (not merge) to preserve df_psp_mapped_cnaf's index, which df_final
# rows below are selected against. Rows without a value (404, error, non-ARS) stay NaN.
allocataire_key = df_psp_mapped_cnaf['allocataire-code_organisme'] + '|' + df_psp_mapped_cnaf['allocataire-matricule']
df_psp_mapped_cnaf['qf_value'] = allocataire_key.map(qf_value_by_allocataire)


In [ ]:
# Quotient familial route: 6-17 ans révolus, household quotient must also clear the threshold.
# The same window already trimmed the qf-batch input above (QF_DOB_MIN/QF_DOB_MAX); re-applied
# here per beneficiary because an eligible household can also hold out-of-window children.
QF_MAX = 700  # quotient familial strictly below this value

mask_qf_dob_start = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date >= QF_DOB_MIN.date()
mask_qf_dob_end = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date <= QF_DOB_MAX.date()
# NaN qf_value (no verdict from qf-batch: 404, error, non-ARS row) compares False here.
mask_qf_value = df_psp_mapped_cnaf['qf_value'] < QF_MAX
mask_qf = mask_qf_value & mask_qf_dob_start & mask_qf_dob_end

df_final_jeune = df_final[mask_qf]


In [ ]:
# AAH route: 16-30 ans révolus - situation already settled from CNAF's ORIGINESELECTION
mask_aah_dob_start = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date >= datetime(1996, 1, 1).date()
mask_aah_dob_end = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date <= datetime(2010, 12, 31).date()
mask_aah = (df_psp_mapped_cnaf['situation'] == 'AAH') & mask_aah_dob_start & mask_aah_dob_end

df_final_aah = df_final[mask_aah]


In [ ]:
# AEEH route: 6-19 ans révolus - no quotient_familial call needed, CNAF already flags it
mask_aeeh_dob_start = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date >= datetime(2007, 1, 1).date()
mask_aeeh_dob_end = pd.to_datetime(df_psp_mapped_cnaf['date_naissance']).dt.date <= datetime(2020, 12, 31).date()
mask_aeeh = (df_psp_mapped_cnaf['situation'] == 'AEEH') & mask_aeeh_dob_start & mask_aeeh_dob_end

df_final_aeeh = df_final[mask_aeeh]


In [ ]:
# Merge QF, AAH and AEEH routes
df_final_jeune_and_aah = pd.concat(
    [df_final_jeune, df_final_aah, df_final_aeeh], ignore_index=True).reset_index(drop=True)


In [ ]:
df_final_jeune_and_aah.loc[:,'date_naissance'] = df_final_jeune_and_aah['date_naissance'].astype(str)

In [ ]:
# output to CSV files
df_final_jeune_and_aah.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [ ]:
print(f"{len(df_final_jeune)} df_final_jeune")
print(f"{len(df_final_aah)} df_final_aah")
print(f"{len(df_final_aeeh)} df_final_aeeh")
print(f"{len(df_final_jeune_and_aah)} jeune, aah and aeeh")
